In [1]:
import importlib
import IPython.display as ipd
import numpy as np
from src.binaural_attn_lightning import BinauralAttentionModule
from pathlib import Path
import os 
import yaml
import tqdm
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [2]:
mdl_ckpt = 'attn_cue_models/binaural_word_task_cue_voiec_and_loc_v02/checkpoints/epoch=0-step=2000-v3.ckpt'

In [3]:
config = yaml.load(open('/om2/user/imgriff/projects/Auditory-Attention/config/binaural_attn/dev_voice_and_loc_cue_001.yaml', 'r'), Loader=yaml.FullLoader)

In [13]:
batch_size = 32
config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0
## Make sure model can run on validation set 
config['num_workers'] = 10
config['hparas']['batch_size'] = batch_size


In [14]:
# class_map =  model.dataset(**config['corpus'], mode='val').class_map()

# ix_to_word = {v:k for k,v in class_map.items()}

In [15]:
# val_loader = model.val_dataloader()

In [16]:
# cue_features, cue_mask_ixs, scene_features, labels = next(iter(val_loader))
# 

In [17]:
# ipd.Audio(cue_features[2], rate=50_000)

In [18]:
# ipd.Audio(scene_features[2], rate=50_000)

In [19]:
# cue_mask_ixs

In [20]:
import torchmetrics
import torch
acc = torchmetrics.Accuracy()


module = BinauralAttentionModule.load_from_checkpoint(checkpoint_path=mdl_ckpt, config=config)
module.freeze()
module = module.cuda()

val_loader = module.val_dataloader()

results = []
n_to_measure = int(len(val_loader) * 0.01)

with torch.no_grad():
    for i, batch in enumerate(tqdm.tqdm(val_loader, total=n_to_measure)):
        if i == n_to_measure:
            break
        cue_features, cue_mask_ixs, scene_features, labels = batch
        # out = module.forward(cue_features.cuda(), scene_features.cuda(), cue_mask_ixs)
        out = module(cue_features.cuda(), scene_features.cuda(), None)

        acc(out.cpu(), labels)
        top_1 = out.softmax(-1).argmax(-1).cpu()
        results.append(np.mean(top_1.numpy() == labels.numpy()))

    

num_classes={'num_words': 800}
Model performing word task
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram
49 files in val concat dataset


100%|██████████| 46/46 [02:10<00:00,  2.84s/it]


In [21]:
acc.compute(), np.mean().round(4)

(tensor(0.4056), 0.4056)

In [22]:
ipd.Audio(scene_features[0], rate=50_000)

In [ ]:
from pytorch_lightning import Trainer

trainer = Trainer(
        limit_train_batches=0,
        limit_val_batches=0.01,
        num_nodes=1,
        gpus=1,
        accelerator="gpu",
    )
module = BinauralAttentionModule.load_from_checkpoint(checkpoint_path=mdl_ckpt, config=config)
trainer.validate(module)


GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


num_classes={'num_words': 800}
Model performing word task


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram
49 files in val concat dataset


Validation: 0it [00:00, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       ACC/val_acc          0.6691576242446899
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'ACC/val_acc': 0.6691576242446899}]